In [1]:
!nvidia-smi

Fri Mar  6 16:15:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes python-chess huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 75.4 MB/s eta 0:00:00:00:010:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.9 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch
import random
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

In [ ]:
login()

In [6]:
MODEL_ID = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = "qwen3-chess"
HF_REPO = "csncsate/LLMChess"
MAX_SAMPLES = 300_000
MAX_LENGTH = 128

In [7]:
raw_dataset = load_dataset(
    "Lichess/chess-position-evaluations",
    split="train",
    streaming=True
)

def preprocess(example):
    fen = example["fen"]
    move = example["line"].split()[0]
    cp = example["cp"]

    if cp is None:
        return None
    if cp < -200:
        return None

    return {"text": f"FEN: {fen}\nMove: {move}"}

samples = []
for example in raw_dataset.shuffle(seed=42, buffer_size=10_000):
    result = preprocess(example)
    if result:
        samples.append(result)
    if len(samples) >= MAX_SAMPLES:
        break

dataset = Dataset.from_list(samples)
print(f"Dataset size: {len(dataset)}")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Dataset size: 300000


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [9]:
for name, _ in model.named_modules():
    print(name)


model
model.embed_tokens
model.layers
model.layers.0
model.layers.0.self_attn
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.self_attn.q_norm
model.layers.0.self_attn.k_norm
model.layers.0.mlp
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.act_fn
model.layers.0.input_layernorm
model.layers.0.post_attention_layernorm
model.layers.1
model.layers.1.self_attn
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.self_attn.q_norm
model.layers.1.self_attn.k_norm
model.layers.1.mlp
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.1.mlp.act_fn
model.layers.1.input_layernorm
model.layers.1.post_attention_layernorm
model.layers.2
model.layers.2.self_attn
model.layers.2.self_attn.q_proj
model.layers.2.self_att

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,293,760 || all params: 753,926,144 || trainable%: 0.3042


In [14]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=100,
    save_steps=1000,
    warmup_steps=200,
    lr_scheduler_type="cosine",
    report_to="none",
    max_length=MAX_LENGTH,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/300000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/300000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/300000 [00:00<?, ? examples/s]

Step,Training Loss
100,2.969943
200,1.530567
300,1.290569
400,1.243796
500,1.214199
600,1.192129
700,1.177363
800,1.165445
900,1.164263
1000,1.157655


TrainOutput(global_step=9375, training_loss=1.0872276896158855, metrics={'train_runtime': 11119.8031, 'train_samples_per_second': 26.979, 'train_steps_per_second': 0.843, 'total_flos': 6.679709714743296e+16, 'train_loss': 1.0872276896158855})

In [15]:
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  13%|#2        |  583kB / 4.60MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpgl8geojw/tokenizer.json:   0%|          | 28.3kB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/csncsate/LLMchess/commit/af53e176ac463eb6918e586713b95e740d4d03fe', commit_message='Upload tokenizer', commit_description='', oid='af53e176ac463eb6918e586713b95e740d4d03fe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/csncsate/LLMchess', endpoint='https://huggingface.co', repo_type='model', repo_id='csncsate/LLMchess'), pr_revision=None, pr_num=None)